In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect('../dataset/nfts.sqlite/nfts.sqlite')

In [3]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)
print(tables)

                                                 name
0                                                nfts
1                                          checkpoint
2                                               mints
3                                           transfers
4   transfer_values_quartile_10_distribution_per_a...
5                                      current_owners
6                               current_market_values
7                          market_values_distribution
8                      transfer_statistics_by_address
9   transfer_values_quantile_10_distribution_per_a...
10  transfer_values_quantile_25_distribution_per_a...
11                                    transfers_mints
12                                 mint_holding_times
13                             transfer_holding_times
14                              ownership_transitions


In [4]:
transfers = pd.read_sql_query(
    "SELECT * FROM transfers LIMIT 10000",
    conn
)
print(transfers.head())
print(transfers.columns.tolist())
print(transfers.dtypes)


                               event_id  \
0  cd816651-56b2-4ed9-887c-c83de732428d   
1  82cc5228-eb80-4e0d-9f6f-e644dec3ab06   
2  6e1f9cc4-d1df-4a6b-972d-a20765beb326   
3  43dc27a7-a72d-4894-809c-e868de05f7ee   
4  47b7839b-9b87-442d-b2c1-9ebedcad8e06   

                                    transaction_hash  block_number  \
0  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   
1  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   
2  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   
3  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   
4  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   

                                  nft_address  \
0  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   
1  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   
2  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   
3  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   
4  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   

                         

In [5]:
ownership = pd.read_sql_query(
    "SELECT * FROM ownership_transitions LIMIT 1000",
    conn
)
print(ownership.columns.tolist())

['from_address', 'to_address', 'num_transitions']


In [6]:
# Cek missing values
print(transfers.isnull().sum())

# Cek range timestamp
print(transfers['timestamp'].min())
print(transfers['timestamp'].max())

# Cek jumlah unique wallet
print(transfers['from_address'].nunique())
print(transfers['to_address'].nunique())

# Cek jumlah transaksi
print(len(transfers))

event_id             0
transaction_hash     0
block_number         0
nft_address          0
token_id             0
from_address         0
to_address           0
transaction_value    0
timestamp            0
dtype: int64
1627776017
1627809639
4090
3695
10000


In [7]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)
print("=== SEMUA TABEL ===")
print(tables)

# Cek schema setiap tabel penting
tabel_penting = [
    'transfers',
    'ownership_transitions', 
    'transfer_statistics_by_address',
    'transfer_holding_times',
    'current_market_values',
    'mints',
    'nfts'
]

for tabel in tabel_penting:
    try:
        schema = pd.read_sql_query(
            f"PRAGMA table_info({tabel})", conn
        )
        print(f"\n=== SCHEMA: {tabel} ===")
        print(schema[['name', 'type']].to_string())
        
        sample = pd.read_sql_query(
            f"SELECT * FROM {tabel} LIMIT 3", conn
        )
        print(f"Sample data:")
        print(sample.head(3))
    except Exception as e:
        print(f"ERROR di {tabel}: {e}")

=== SEMUA TABEL ===
                                                 name
0                                                nfts
1                                          checkpoint
2                                               mints
3                                           transfers
4   transfer_values_quartile_10_distribution_per_a...
5                                      current_owners
6                               current_market_values
7                          market_values_distribution
8                      transfer_statistics_by_address
9   transfer_values_quantile_10_distribution_per_a...
10  transfer_values_quantile_25_distribution_per_a...
11                                    transfers_mints
12                                 mint_holding_times
13                             transfer_holding_times
14                              ownership_transitions

=== SCHEMA: transfers ===
                name     type
0           event_id     TEXT
1   transaction_hash     TEXT

In [8]:
# Load transfers — ini inti semua analisis
print("Loading transfers...")
transfers = pd.read_sql_query("""
    SELECT 
        transaction_hash,
        block_number,
        timestamp,
        nft_address,
        token_id,
        from_address,
        to_address,
        transaction_value
    FROM transfers
    LIMIT 500000
""", conn)

print(f"Shape: {transfers.shape}")
print(f"Columns: {transfers.columns.tolist()}")
print(f"Dtypes:\n{transfers.dtypes}")
print(f"Sample:\n{transfers.head()}")
print(f"Null values:\n{transfers.isnull().sum()}")

# Cek timestamp
print(f"\nTimestamp min: {transfers['timestamp'].min()}")
print(f"Timestamp max: {transfers['timestamp'].max()}")
print(f"Unique wallets: {transfers['from_address'].nunique()}")
print(f"Unique NFTs: {transfers['token_id'].nunique()}")

Loading transfers...
Shape: (500000, 8)
Columns: ['transaction_hash', 'block_number', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value']
Dtypes:
transaction_hash         str
block_number           int64
timestamp              int64
nft_address              str
token_id                 str
from_address             str
to_address               str
transaction_value    float64
dtype: object
Sample:
                                    transaction_hash  block_number  \
0  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   
1  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   
2  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   
3  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   
4  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...      12936373   

    timestamp                                 nft_address  \
0  1627776481  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   
1  1627776481  0x6

In [9]:
import numpy as np
from itertools import combinations

def label_wash_trading(transfers_df, 
                        max_cycle_hours=24,  # max durasi siklus
                        max_holding_hours=24, # max holding time
                        min_price_increase=0.1): # min kenaikan harga
    """
    Label transaksi sebagai wash trading berdasarkan 3 kriteria:
    1. Ada siklus A→B→...→A
    2. Siklus terjadi dalam waktu singkat (max_cycle_hours)
    3. Ada kenaikan harga dalam siklus
    
    Referensi: Von Wachter et al. (2022), La Morgia et al. (2023)
    """
    
    df = transfers_df.copy()
    
    # Pastikan timestamp dalam format datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')
    
    # Inisialisasi label
    df['is_wash_trading'] = 0
    
    # Group by NFT (per token_id + nft_address)
    wash_hashes = set()
    
    for (nft_addr, token_id), group in df.groupby(
        ['nft_address', 'token_id']
    ):
        if len(group) < 2:
            continue
            
        group = group.sort_values('timestamp').reset_index(drop=True)
        
        # Cek siklus: apakah ada wallet yang muncul sebagai
        # from_address dan to_address dalam window waktu tertentu
        for i in range(len(group)):
            for j in range(i+1, len(group)):
                
                time_diff = (
                    group.loc[j, 'timestamp'] - 
                    group.loc[i, 'timestamp']
                ).total_seconds() / 3600  # convert ke jam
                
                if time_diff > max_cycle_hours:
                    break
                    
                # Cek apakah ini siklus (from_i == to_j atau sebaliknya)
                if (group.loc[i, 'from_address'] == 
                    group.loc[j, 'to_address']):
                    
                    # Ada siklus dalam max_cycle_hours jam
                    # Cek price increase
                    price_i = group.loc[i, 'transaction_value'] or 0
                    price_j = group.loc[j, 'transaction_value'] or 0
                    
                    if price_i > 0 and price_j > price_i:
                        price_increase = (price_j - price_i) / price_i
                        if price_increase >= min_price_increase:
                            # Ini wash trading!
                            wash_hashes.add(
                                group.loc[i, 'transaction_hash']
                            )
                            wash_hashes.add(
                                group.loc[j, 'transaction_hash']
                            )
    
    df.loc[
        df['transaction_hash'].isin(wash_hashes), 
        'is_wash_trading'
    ] = 1
    
    return df

# Test dengan sample kecil dulu
sample = transfers.head(10000)
labeled = label_wash_trading(sample)
print(f"Total transaksi: {len(labeled)}")
print(f"Wash trading: {labeled['is_wash_trading'].sum()}")
print(f"Ratio: {labeled['is_wash_trading'].mean():.3f}")

Total transaksi: 10000
Wash trading: 0
Ratio: 0.000


In [10]:
# Jalankan ini untuk diagnosa
print("=== DIAGNOSA DATA ===")

# 1. Cek transaction_value
print("\n1. Transaction Value Stats:")
print(transfers['transaction_value'].describe())
print(f"Null values: {transfers['transaction_value'].isnull().sum()}")
print(f"Zero values: {(transfers['transaction_value'] == 0).sum()}")
print(f"Non-zero: {(transfers['transaction_value'] > 0).sum()}")

# 2. Cek apakah ada wallet yang muncul di from DAN to
from_wallets = set(transfers['from_address'].dropna())
to_wallets = set(transfers['to_address'].dropna())
overlap = from_wallets.intersection(to_wallets)
print(f"\n2. Wallet overlap (muncul di from & to): {len(overlap)}")

# 3. Cek apakah ada siklus sederhana manual
# ambil 1 token_id yang punya banyak transaksi
top_tokens = transfers.groupby(
    ['nft_address', 'token_id']
).size().sort_values(ascending=False).head(10)
print(f"\n3. Top tokens by transaction count:")
print(top_tokens)

# 4. Cek satu token secara manual
top_token = top_tokens.index[0]
token_sample = transfers[
    (transfers['nft_address'] == top_token[0]) & 
    (transfers['token_id'] == top_token[1])
].sort_values('timestamp')
print(f"\n4. Sample transaksi untuk token paling aktif:")
print(token_sample[['timestamp','from_address','to_address','transaction_value']])

# 5. Cek format timestamp
print(f"\n5. Timestamp sample:")
print(transfers['timestamp'].head(10))
print(f"Dtype: {transfers['timestamp'].dtype}")

=== DIAGNOSA DATA ===

1. Transaction Value Stats:
count    5.000000e+05
mean     4.405352e+17
std      2.825930e+18
min      0.000000e+00
25%      0.000000e+00
50%      6.250000e+16
75%      2.290000e+17
max      4.690000e+20
Name: transaction_value, dtype: float64
Null values: 0
Zero values: 161037
Non-zero: 338963

2. Wallet overlap (muncul di from & to): 32595

3. Top tokens by transaction count:
nft_address                                 token_id
0xACd3CF818EFe8ddce84C585ddCB147c4C844D3b3  0           205
0x810Ac3aeFc34806DCdaF1493bfe3bE32759ED262  64           23
0x60F80121C31A0d46B5279700f9DF786054aa5eE5  730303       16
0x5d99371A4297DFD301a1F22EfF22A7e0ED9B4482  55           15
                                            18           15
0x0062b396597FE833Ce110672F81D411f4e5042F0  3            14
0x5d99371A4297DFD301a1F22EfF22A7e0ED9B4482  74           11
0x11595fFB2D3612d810612e34Bc1C2E6D6de55d26  1152         11
0x27b4bC90fBE56f02Ef50f2E2f79D7813Aa8941A7  7530         11
0x1

In [11]:
import pandas as pd
import numpy as np

def label_wash_trading_v2(transfers_df,
                           max_cycle_hours=24,
                           min_pair_interactions=2,
                           min_token_turnover=5):
    """
    Labeling wash trading dengan 3 strategi:
    
    Strategi 1: Direct/Multi-hop Cycle Detection
    - A→B→A atau A→B→C→A dalam window waktu singkat
    - Referensi: Von Wachter et al. (2022)
    
    Strategi 2: Repeated Pair Interaction
    - Wallet A dan B saling transfer NFT yang sama
      lebih dari N kali
    - Referensi: La Morgia et al. (2023)
    
    Strategi 3: High Turnover + Zero Value
    - NFT berpindah tangan cepat dengan value = 0
    - Referensi: Hemenway Falk et al. (2024)
    """
    
    df = transfers_df.copy()
    
    # Fix 1: Convert Unix timestamp ke datetime
    if df['timestamp'].dtype in ['int64', 'float64']:
        df['timestamp'] = pd.to_datetime(
            df['timestamp'], unit='s'
        )
    else:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['is_wash_trading'] = 0
    wash_hashes = set()
    
    print("Processing labeling...")
    
    # =============================================
    # STRATEGI 1: Cycle Detection per NFT Token
    # =============================================
    print("Strategy 1: Cycle Detection...")
    
    for (nft_addr, token_id), group in df.groupby(
        ['nft_address', 'token_id']
    ):
        if len(group) < 2:
            continue
            
        group = group.sort_values('timestamp').reset_index(drop=True)
        addresses = group[['from_address', 'to_address', 
                           'timestamp', 'transaction_hash']].values
        
        for i in range(len(addresses)):
            for j in range(i+1, len(addresses)):
                
                # Hitung time diff dalam jam
                time_diff = (
                    addresses[j][2] - addresses[i][2]
                ).total_seconds() / 3600
                
                if time_diff > max_cycle_hours:
                    break
                
                # Direct cycle: from_i == to_j
                # (A→B lalu B→A, A kembali dapat NFT)
                if addresses[i][0] == addresses[j][1]:
                    wash_hashes.add(addresses[i][3])
                    wash_hashes.add(addresses[j][3])
                
                # Juga cek: to_i == from_j (chain normal)
                # dan from_i pernah jadi to di transaksi lain
    
    print(f"  Cycle detection found: {len(wash_hashes)} hashes")
    
    # =============================================
    # STRATEGI 2: Repeated Pair Interaction
    # =============================================
    print("Strategy 2: Repeated Pair Interaction...")
    
    # Buat pair key (wallet_a, wallet_b) tidak peduli arahnya
    df['wallet_pair'] = df.apply(
        lambda r: tuple(sorted([
            str(r['from_address']), 
            str(r['to_address'])
        ])),
        axis=1
    )
    
    # Hitung interaksi per pair per NFT
    pair_nft_counts = df.groupby(
        ['wallet_pair', 'nft_address', 'token_id']
    ).size().reset_index(name='interaction_count')
    
    # Pair yang sering transaksi NFT yang sama = suspicious
    suspicious_pairs = pair_nft_counts[
        pair_nft_counts['interaction_count'] >= min_pair_interactions
    ]
    
    for _, row in suspicious_pairs.iterrows():
        mask = (
            (df['wallet_pair'] == row['wallet_pair']) &
            (df['nft_address'] == row['nft_address']) &
            (df['token_id'] == row['token_id'])
        )
        wash_hashes.update(
            df[mask]['transaction_hash'].tolist()
        )
    
    print(f"  After repeated pair: {len(wash_hashes)} hashes")
    
    # =============================================
    # STRATEGI 3: High Turnover + Zero Value
    # =============================================
    print("Strategy 3: High Turnover + Zero Value...")
    
    # NFT yang banyak berpindah tangan dengan value = 0
    zero_value_transfers = df[df['transaction_value'] == 0]
    
    token_turnover = zero_value_transfers.groupby(
        ['nft_address', 'token_id']
    ).size().reset_index(name='turnover_count')
    
    high_turnover = token_turnover[
        token_turnover['turnover_count'] >= min_token_turnover
    ]
    
    for _, row in high_turnover.iterrows():
        mask = (
            (df['nft_address'] == row['nft_address']) &
            (df['token_id'] == row['token_id']) &
            (df['transaction_value'] == 0)
        )
        wash_hashes.update(
            df[mask]['transaction_hash'].tolist()
        )
    
    print(f"  After high turnover: {len(wash_hashes)} hashes")
    
    # Apply labels
    df.loc[
        df['transaction_hash'].isin(wash_hashes),
        'is_wash_trading'
    ] = 1
    
    # Cleanup
    df = df.drop(columns=['wallet_pair'])
    
    return df

# Test dengan 500k data
print("Loading data...")
transfers = pd.read_sql_query("""
    SELECT 
        transaction_hash,
        block_number,
        timestamp,
        nft_address,
        token_id,
        from_address,
        to_address,
        transaction_value
    FROM transfers
    LIMIT 500000
""", conn)

print("Labeling...")
labeled = label_wash_trading_v2(transfers)

print(f"\n=== HASIL ===")
print(f"Total transaksi: {len(labeled):,}")
print(f"Wash trading: {labeled['is_wash_trading'].sum():,}")
print(f"Normal: {(labeled['is_wash_trading']==0).sum():,}")
print(f"Ratio wash trading: {labeled['is_wash_trading'].mean():.3%}")

# Cek per strategi
print("\nSample wash trading transactions:")
print(labeled[labeled['is_wash_trading']==1][
    ['timestamp','from_address','to_address',
     'transaction_value','is_wash_trading']
].head(10))

Loading data...
Labeling...
Processing labeling...
Strategy 1: Cycle Detection...
  Cycle detection found: 5083 hashes
Strategy 2: Repeated Pair Interaction...
  After repeated pair: 7977 hashes
Strategy 3: High Turnover + Zero Value...
  After high turnover: 8464 hashes

=== HASIL ===
Total transaksi: 500,000
Wash trading: 15,125
Normal: 484,875
Ratio wash trading: 3.025%

Sample wash trading transactions:
              timestamp                                from_address  \
68  2021-08-01 00:04:25  0xDf66A3C9122C031FFDe5aa4300d53869ebBc6a1d   
71  2021-08-01 00:04:25  0x67BDcD02705CEcf08Cb296394DB7d6Ed00A496F9   
164 2021-08-01 00:08:45  0xc3f733ca98E0daD0386979Eb96fb1722A1A05E69   
165 2021-08-01 00:08:45  0xB741401653dbAa4b869AFF4327B0B5daf7750040   
166 2021-08-01 00:08:45  0xc3f733ca98E0daD0386979Eb96fb1722A1A05E69   
167 2021-08-01 00:08:45  0xB741401653dbAa4b869AFF4327B0B5daf7750040   
168 2021-08-01 00:08:45  0xc3f733ca98E0daD0386979Eb96fb1722A1A05E69   
169 2021-08-01 00:08:

In [12]:
# Alamat yang harus di-exclude dari wash trading detection
EXCLUDE_ADDRESSES = {
    '0x0000000000000000000000000000000000000000',  # Burn/null address
    '0x000000000000000000000000000000000000dead',  # Dead address
}

def label_wash_trading_v3(transfers_df,
                           max_cycle_hours=24,
                           min_pair_interactions=3,  # naikkan threshold
                           min_token_turnover=5):
    
    df = transfers_df.copy()
    
    # Fix timestamp
    if df['timestamp'].dtype in ['int64', 'float64']:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['is_wash_trading'] = 0
    wash_hashes = set()
    
    # FILTER: Hapus transaksi yang melibatkan burn/null address
    df_clean = df[
        ~df['from_address'].str.lower().isin(
            [a.lower() for a in EXCLUDE_ADDRESSES]
        ) &
        ~df['to_address'].str.lower().isin(
            [a.lower() for a in EXCLUDE_ADDRESSES]
        )
    ].copy()
    
    print(f"After filtering burn addresses: {len(df_clean):,} rows")
    print(f"Removed: {len(df) - len(df_clean):,} rows")
    
    # =============================================
    # STRATEGI 1: Direct Cycle A→B→A
    # Referensi: Von Wachter et al. (2022)
    # =============================================
    print("\nStrategy 1: Direct Cycle Detection...")
    cycle_count = 0
    
    for (nft_addr, token_id), group in df_clean.groupby(
        ['nft_address', 'token_id']
    ):
        if len(group) < 2:
            continue
        
        group = group.sort_values('timestamp').reset_index(drop=True)
        
        for i in range(len(group)):
            for j in range(i+1, len(group)):
                
                time_diff = (
                    group.loc[j, 'timestamp'] - 
                    group.loc[i, 'timestamp']
                ).total_seconds() / 3600
                
                if time_diff > max_cycle_hours:
                    break
                
                # Direct cycle: A→B lalu B→A
                from_i = group.loc[i, 'from_address']
                to_i   = group.loc[i, 'to_address']
                from_j = group.loc[j, 'from_address']
                to_j   = group.loc[j, 'to_address']
                
                # A→B→A pattern
                if from_i == to_j and to_i == from_j:
                    wash_hashes.add(group.loc[i, 'transaction_hash'])
                    wash_hashes.add(group.loc[j, 'transaction_hash'])
                    cycle_count += 1
    
    print(f"  Direct cycles found: {cycle_count}")
    print(f"  Hashes so far: {len(wash_hashes)}")
    
    # =============================================
    # STRATEGI 2: Repeated Pair Interaction
    # Referensi: La Morgia et al. (2023)
    # Hanya flag kalau pair saling tukar NFT yang SAMA
    # lebih dari N kali - ini lebih ketat dari v2
    # =============================================
    print("\nStrategy 2: Repeated Pair + Same NFT...")
    
    # Pair interaction per NFT token spesifik
    df_clean['wallet_pair'] = df_clean.apply(
        lambda r: tuple(sorted([
            str(r['from_address']).lower(),
            str(r['to_address']).lower()
        ])),
        axis=1
    )
    
    pair_token_counts = df_clean.groupby(
        ['wallet_pair', 'nft_address', 'token_id']
    ).size().reset_index(name='count')
    
    # Hanya flag pair yang transfer NFT YANG SAMA >= N kali
    # Ini lebih spesifik dan less false positive
    suspicious = pair_token_counts[
        pair_token_counts['count'] >= min_pair_interactions
    ]
    
    pair2_count = 0
    for _, row in suspicious.iterrows():
        mask = (
            (df_clean['wallet_pair'] == row['wallet_pair']) &
            (df_clean['nft_address'] == row['nft_address']) &
            (df_clean['token_id'] == row['token_id'])
        )
        new_hashes = set(df_clean[mask]['transaction_hash'].tolist())
        new_found = new_hashes - wash_hashes
        wash_hashes.update(new_found)
        pair2_count += len(new_found)
    
    print(f"  New hashes from repeated pair: {pair2_count}")
    print(f"  Hashes so far: {len(wash_hashes)}")
    
    # =============================================
    # STRATEGI 3: High Turnover dengan Rapid Trading
    # Referensi: Hemenway Falk et al. (2024)
    # NFT yang berpindah cepat + zero value
    # =============================================
    print("\nStrategy 3: Rapid High Turnover...")
    
    # Hitung interval antar transfer per NFT
    df_sorted = df_clean.sort_values(
        ['nft_address', 'token_id', 'timestamp']
    )
    df_sorted['prev_timestamp'] = df_sorted.groupby(
        ['nft_address', 'token_id']
    )['timestamp'].shift(1)
    
    df_sorted['interval_hours'] = (
        df_sorted['timestamp'] - df_sorted['prev_timestamp']
    ).dt.total_seconds() / 3600
    
    # NFT yang sering berpindah dalam interval sangat cepat
    # dan dengan value = 0
    rapid_zero = df_sorted[
        (df_sorted['interval_hours'] < 1) &  # < 1 jam antar transfer
        (df_sorted['transaction_value'] == 0)
    ]
    
    # Hitung berapa kali per NFT
    rapid_counts = rapid_zero.groupby(
        ['nft_address', 'token_id']
    ).size().reset_index(name='rapid_count')
    
    suspicious_rapid = rapid_counts[
        rapid_counts['rapid_count'] >= min_token_turnover
    ]
    
    strat3_count = 0
    for _, row in suspicious_rapid.iterrows():
        mask = (
            (df_sorted['nft_address'] == row['nft_address']) &
            (df_sorted['token_id'] == row['token_id']) &
            (df_sorted['interval_hours'] < 1) &
            (df_sorted['transaction_value'] == 0)
        )
        new_hashes = set(
            df_sorted[mask]['transaction_hash'].tolist()
        )
        new_found = new_hashes - wash_hashes
        wash_hashes.update(new_found)
        strat3_count += len(new_found)
    
    print(f"  New hashes from rapid turnover: {strat3_count}")
    print(f"  Total hashes: {len(wash_hashes)}")
    
    # Apply labels (hanya ke df original, termasuk yang ke burn address)
    df.loc[
        df['transaction_hash'].isin(wash_hashes),
        'is_wash_trading'
    ] = 1
    
    # Cleanup
    if 'wallet_pair' in df.columns:
        df = df.drop(columns=['wallet_pair'])
    
    return df, df_sorted  # return df_sorted untuk debug

# =============================================
# RUN
# =============================================
labeled, df_debug = label_wash_trading_v3(transfers)

print(f"\n=== HASIL FINAL ===")
print(f"Total transaksi: {len(labeled):,}")
print(f"Wash trading: {labeled['is_wash_trading'].sum():,}")
print(f"Normal: {(labeled['is_wash_trading']==0).sum():,}")
print(f"Ratio: {labeled['is_wash_trading'].mean():.3%}")

# Sanity check — pastikan tidak ada burn address di wash trading
wt_sample = labeled[labeled['is_wash_trading']==1]
burn_in_wt = wt_sample[
    wt_sample['to_address'] == 
    '0x0000000000000000000000000000000000000000'
]
print(f"\nBurn address in wash trading labels: {len(burn_in_wt)}")
print("(Harusnya 0 atau sangat sedikit)")

# Lihat contoh yang lebih masuk akal
print(f"\nSample wash trading (non-zero value):")
wt_nonzero = labeled[
    (labeled['is_wash_trading']==1) & 
    (labeled['transaction_value'] > 0)
]
print(f"Wash trading dengan value > 0: {len(wt_nonzero):,}")
print(wt_nonzero[['timestamp','from_address','to_address',
                   'transaction_value']].head(5))

After filtering burn addresses: 481,135 rows
Removed: 18,865 rows

Strategy 1: Direct Cycle Detection...
  Direct cycles found: 2710
  Hashes so far: 3892

Strategy 2: Repeated Pair + Same NFT...
  New hashes from repeated pair: 238
  Hashes so far: 4130

Strategy 3: Rapid High Turnover...
  New hashes from rapid turnover: 170
  Total hashes: 4300

=== HASIL FINAL ===
Total transaksi: 500,000
Wash trading: 5,869
Normal: 494,131
Ratio: 1.174%

Burn address in wash trading labels: 45
(Harusnya 0 atau sangat sedikit)

Sample wash trading (non-zero value):
Wash trading dengan value > 0: 330
               timestamp                                from_address  \
679  2021-08-01 00:34:15  0xb1690C08E213a35Ed9bAb7B318DE14420FB57d8C   
1234 2021-08-01 01:02:39  0x67BDcD02705CEcf08Cb296394DB7d6Ed00A496F9   
3395 2021-08-01 02:44:25  0xb1690C08E213a35Ed9bAb7B318DE14420FB57d8C   
6878 2021-08-01 05:43:50  0x480eab2151a0446c77119Ca6C5777EC531bc1814   
7067 2021-08-01 05:55:57  0x67BDcD02705CEcf08C

In [13]:
# =============================================
# STEP 2: PREPROCESSING
# =============================================

# Fix timestamp
transfers['timestamp'] = pd.to_datetime(
    transfers['timestamp'], unit='s'
)

# Filter hanya sales (value > 0)
# Align dengan Liu et al. yang hanya count sales
sales = transfers[transfers['transaction_value'] > 0].copy()
print(f"Sales only (value > 0): {len(sales):,}")

# Filter burn/null address
BURN = {
    '0x0000000000000000000000000000000000000000',
    '0x000000000000000000000000000000000000dead'
}
sales_clean = sales[
    ~sales['from_address'].str.lower().isin(
        {a.lower() for a in BURN}
    ) &
    ~sales['to_address'].str.lower().isin(
        {a.lower() for a in BURN}
    )
].copy().reset_index(drop=True)

print(f"After burn filter: {len(sales_clean):,}")

# =============================================
# STEP 3: LABELING — Liu et al. (2023)
# arxiv:2305.01543
# Rule: wallet A jual NFT X, lalu beli kembali
# NFT X yang sama dalam 30 hari
# =============================================

def label_liu_method(df, max_days=30):
    
    df = df.copy().sort_values('timestamp').reset_index(drop=True)
    df['is_wash_trading'] = 0
    wash_hashes = set()
    max_seconds = max_days * 24 * 3600
    
    total_groups = df.groupby(['nft_address', 'token_id']).ngroups
    print(f"Processing {total_groups:,} unique NFT tokens...")
    
    processed = 0
    for (nft_addr, token_id), group in df.groupby(
        ['nft_address', 'token_id']
    ):
        if len(group) < 2:
            continue
        
        group = group.sort_values(
            'timestamp'
        ).reset_index(drop=True)
        
        for i in range(len(group)):
            seller_i  = group.loc[i, 'from_address'].lower()
            hash_i    = group.loc[i, 'transaction_hash']
            ts_i      = group.loc[i, 'timestamp']
            
            for j in range(i+1, len(group)):
                ts_j     = group.loc[j, 'timestamp']
                time_diff = (ts_j - ts_i).total_seconds()
                
                # Stop kalau sudah lewat threshold
                if time_diff > max_seconds:
                    break
                
                buyer_j = group.loc[j, 'to_address'].lower()
                hash_j  = group.loc[j, 'transaction_hash']
                
                # Wallet yang jual = wallet yang beli kembali
                if seller_i == buyer_j:
                    wash_hashes.add(hash_i)
                    wash_hashes.add(hash_j)
        
        processed += 1
        if processed % 10000 == 0:
            print(f"  Progress: {processed:,}/{total_groups:,}")
    
    df.loc[
        df['transaction_hash'].isin(wash_hashes),
        'is_wash_trading'
    ] = 1
    
    return df

# Run labeling dengan threshold 30 hari
print("\nRunning labeling (Liu et al. 30-day threshold)...")
labeled = label_liu_method(sales_clean, max_days=30)

# =============================================
# STEP 4: HASIL
# =============================================
print(f"\n=== HASIL LABELING ===")
print(f"Total sales: {len(labeled):,}")
print(f"Wash trading: {labeled['is_wash_trading'].sum():,}")
print(f"Normal: {(labeled['is_wash_trading']==0).sum():,}")
print(f"Ratio: {labeled['is_wash_trading'].mean():.3%}")

# Bandingkan beberapa threshold
print(f"\n=== PERBANDINGAN THRESHOLD ===")
for days in [7, 14, 30, 60]:
    labeled_test = label_liu_method(sales_clean, max_days=days)
    wt = labeled_test['is_wash_trading'].sum()
    ratio = labeled_test['is_wash_trading'].mean()
    print(f"{days:3d} hari: {wt:6,} wash trades ({ratio:.3%})")

Sales only (value > 0): 338,963
After burn filter: 337,233

Running labeling (Liu et al. 30-day threshold)...
Processing 286,846 unique NFT tokens...
  Progress: 10,000/286,846
  Progress: 20,000/286,846
  Progress: 30,000/286,846
  Progress: 40,000/286,846

=== HASIL LABELING ===
Total sales: 337,233
Wash trading: 574
Normal: 336,659
Ratio: 0.170%

=== PERBANDINGAN THRESHOLD ===
Processing 286,846 unique NFT tokens...
  Progress: 10,000/286,846
  Progress: 20,000/286,846
  Progress: 30,000/286,846
  Progress: 40,000/286,846
  7 hari:    570 wash trades (0.169%)
Processing 286,846 unique NFT tokens...
  Progress: 10,000/286,846
  Progress: 20,000/286,846
  Progress: 30,000/286,846
  Progress: 40,000/286,846
 14 hari:    574 wash trades (0.170%)
Processing 286,846 unique NFT tokens...
  Progress: 10,000/286,846
  Progress: 20,000/286,846
  Progress: 30,000/286,846
  Progress: 40,000/286,846
 30 hari:    574 wash trades (0.170%)
Processing 286,846 unique NFT tokens...
  Progress: 10,000/